# SAC Analysis Notebook

This notebook only configures parameters and calls reusable modules in `src/`.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import PipelineParams, list_events, process_event
from src.plotting import (
    PlotParams,
    SaveOptions,
    plot_waveform,
    plot_spectrogram,
    plot_lofar,
    plot_azimuth_spectrogram,
    plot_azimuth_stability,
    plot_azimuth_confidence_mask,
)

In [ ]:
# 所有参数都在这里修改
data_dir = str(PROJECT_ROOT / "data" / "examples")  # 文件存储路径，相对路径即可
window_length_s = 5.0  # 时窗长度
overlap = 0.75  # 重叠度
selected_band = (150.0, 400.0)  # 绘图的频率范围 e.g. (150.0, 400.0) 或 None
time_slice_s = None#(2150.0, 2500.0)  # 切分的时间范围 e.g. (600.0, 780.0) 或 None

# 预处理开关（默认开启）
enable_demean = True #去均值
enable_detrend = True#去均势
enable_bandpass = True#带通滤波
preprocess_band = (150.0,400.0)  # 带通滤波范围，自选 或者 None(代表2.0, 0.8*Nyquist)
apply_orientation = True #是否矫正仪器方位角到北向
orientation_deg = 0.0  # 方位校正角度，逆时针为正

#这三个参数一般不用动
stability_window = 5  # 调节时间稳定性方位角谱的计算窗口
stability_step = 2  # 调节时间稳定性方位角谱的计算步长
confidence_threshold = 0.7  # 调节置信度遮罩阈值

save_figures = True #是否输出结果图
output_dir = PROJECT_ROOT / 'results' / 'figures'  # 结果图保存路径
formats = ('png', 'pdf')

In [ ]:
events = list_events(data_dir)
print(f'Found {len(events)} complete events')
for i, ev in enumerate(events):
    print(i, ev.event_id)

In [ ]:
event_idx = 0 #展示第几个文件结果，上面这一步输出的文献里选择序号
bundle = events[event_idx]

params = PipelineParams(
    data_dir=data_dir,
    window_length_s=window_length_s,
    overlap=overlap,
    selected_band=selected_band,
    time_slice_s=time_slice_s,
    enable_demean=enable_demean,
    enable_detrend=enable_detrend,
    enable_bandpass=enable_bandpass,
    preprocess_band=preprocess_band,
    apply_orientation=apply_orientation,
    orientation_deg=orientation_deg,
    stability_window=stability_window,
    stability_step=stability_step,
    confidence_threshold=confidence_threshold,
)
res = process_event(bundle, params)

print('Filename:', res['event_id'])
print('Selected band:', res['selected_band'])
print('Band candidates:', res['band_info']['candidates'])
print('Time slice:', res['time_slice_s'])
print('Crop info:', res['crop_info'])
print('Preprocess:', res['preprocess_report'])

绘图部分

In [ ]:
plot_params = PlotParams(
    freq_min=res['selected_band'][0],
    freq_max=res['selected_band'][1],
)
save_opts = SaveOptions(
    save=save_figures,
    outdir=output_dir,
    event_id=res['event_id'],
    formats=formats,
)
component = 'BHZ'  # 波形绘制哪个分量 比如BH1 / BH2 / BHZ / HYD

In [ ]:
#绘制波形
fig, _ = plot_waveform(res['t_sec'], res['signals'][component], component, plot_params, save_opts, normalize=True)
plt.show()

In [ ]:
#绘制时频谱
fig, _ = plot_spectrogram(res['t_spec'], res['f_hz'], res['spectrogram_db'][component], component, plot_params, save_opts)
plt.show()

In [ ]:
#绘制Lofar谱（即增强时间方向上连续性的时频谱）
fig, _ = plot_lofar(res['t_spec'], res['f_hz'], res['lofar'][component], component, plot_params, save_opts)
plt.show()

In [ ]:
#绘制方位角谱（根据四分量计算、颜色代表信号方向，不代表强弱）
fig, _ = plot_azimuth_spectrogram(res['t_spec'], res['f_hz'], res['azimuth_deg'], plot_params, save_opts)
plt.show()

In [ ]:
#绘制方位角遮罩谱（根据置信度谱强度设置遮罩、置信度弱的颜色变淡）
#下图为置信度谱（计算四分量能量一致性和方位角谱的幅值，加权得到[0.6,0.4]）
fig, _ = plot_azimuth_confidence_mask(
    res['t_spec'],
    res['f_hz'],
    res['azimuth_masked'],
    res['confidence'],
    threshold=confidence_threshold,
    plot_params=plot_params,
    save_opts=save_opts,
)
plt.show()

#绘制方位角稳定性谱，能量越强稳定性越好
fig, _ = plot_azimuth_stability(res['t_spec'], res['f_hz'], res['azimuth_stability'], plot_params, save_opts)
plt.show()

## TODO
- Structured result tables/exports
- Feature extraction
- Truth-based validation
- Batch processing workflow
- GUI (`app.py`)